# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdulm111/ML-Assignement01/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [15]:
import duckdb, numpy as np, os
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute("CREATE SECRET (TYPE huggingface, TOKEN ?)", [HF_TOKEN])

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"

con.execute(f"""
    CREATE OR REPLACE TABLE march_facts AS
    SELECT * FROM read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')
""")
print("march_facts loaded.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

march_facts loaded.


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Signal check 1 — CTR vs. position (behind `needs_ctr_fix`):**
Mean CTR falls steadily from top_3 (0.354%, n=381,023) down to deep (0.040%,
n=60,821) across all five position tiers, each backed by 60k+ rows. This is what I confirmed.

**Signal check 2 — volume behind `is_quick_win`: **
Better-positioned pages do carry more volume, but 83.2% of quick-win-band
pages (957,239 of 1,150,714) sit below a 100-impression floor  too thin to
trust. Position alone doesn't guarantee enough volume to act on.

**Signal check 3 — does content_type matter beyond position, added after
joining `dim_content`?** This is what I confirmed earlier on the starter CSV: content_type
shifted expected CTR by up to ~0.57 points beyond what tier alone explained,
and cut unexplained CTR variance by 3.4%. A flat tier-only average unfairly
compares a naturally-hot `feedly article` against a naturally-cooler
`comparison article` ranked in the same spot  so "expected CTR" is
computed per (position_tier, content_type) pair, not tier alone.

**The rule, in plain words:** A page is worth a title/description review if
it's live and published, ranks somewhere on the first few results pages,
got enough visits this month to trust its click rate, and its actual click
rate is worse than other pages of the same content type ranking in the same
general spot. The bigger the shortfall and the more visits it's wasting,
the higher it's ranked.

**Reason code:** every flagged page carries one  `ctr_below_expected`.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Signal Check 1
con.sql("""
    SELECT
        CASE
            WHEN gsc_avg_position = 0 THEN 'no_data'
            WHEN gsc_avg_position <= 3 THEN 'top_3'
            WHEN gsc_avg_position <= 10 THEN 'page_1'
            WHEN gsc_avg_position <= 20 THEN 'striking'
            WHEN gsc_avg_position <= 50 THEN 'page_3_5'
            ELSE 'deep'
        END AS position_tier,
        COUNT(*) AS n,
        ROUND(AVG(CAST(gsc_clicks AS DOUBLE) / NULLIF(gsc_impressions,0)) * 100, 3) AS mean_ctr_pct
    FROM march_facts
    WHERE gsc_data_available IS TRUE AND gsc_impressions >= 10
    GROUP BY 1
    ORDER BY mean_ctr_pct DESC
""").show()

┌───────────────┬────────┬──────────────┐
│ position_tier │   n    │ mean_ctr_pct │
│    varchar    │ int64  │    double    │
├───────────────┼────────┼──────────────┤
│ top_3         │ 381023 │        0.354 │
│ page_1        │ 955680 │        0.329 │
│ striking      │ 358200 │        0.261 │
│ page_3_5      │ 384857 │        0.145 │
│ no_data       │   6948 │        0.128 │
│ deep          │  60821 │         0.04 │
└───────────────┴────────┴──────────────┘



In [17]:
# Signal check 2
con.sql("""
    SELECT
        CASE WHEN gsc_avg_position <= 10 THEN 'top_page1'
             WHEN gsc_avg_position <= 50 THEN 'quick_win_band'
             ELSE 'other' END AS band,
        CASE WHEN gsc_impressions < 100 THEN 'low_vol (<100)'
             WHEN gsc_impressions < 1000 THEN 'mid_vol (100-999)'
             ELSE 'high_vol (1000+)' END AS volume_bucket,
        COUNT(*) AS n
    FROM march_facts
    WHERE gsc_data_available IS TRUE AND gsc_avg_position > 0
    GROUP BY 1, 2
    ORDER BY 1, 2
""").show()

┌────────────────┬───────────────────┬─────────┐
│      band      │   volume_bucket   │    n    │
│    varchar     │      varchar      │  int64  │
├────────────────┼───────────────────┼─────────┤
│ other          │ high_vol (1000+)  │     222 │
│ other          │ low_vol (<100)    │  272303 │
│ other          │ mid_vol (100-999) │    4338 │
│ quick_win_band │ high_vol (1000+)  │    9332 │
│ quick_win_band │ low_vol (<100)    │  957239 │
│ quick_win_band │ mid_vol (100-999) │  184143 │
│ top_page1      │ high_vol (1000+)  │   22863 │
│ top_page1      │ low_vol (<100)    │ 1579797 │
│ top_page1      │ mid_vol (100-999) │  417635 │
└────────────────┴───────────────────┴─────────┘



## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

*Score, rank, write work/outputs/baseline_action_score.csv  now joined
against `dim_content` for content_type-aware expected CTR.*

In [18]:
monthly = con.sql("""
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_impressions) AS impressions_month,
        SUM(gsc_clicks) AS clicks_month,
        SUM(gsc_avg_position * gsc_impressions) / NULLIF(SUM(gsc_impressions), 0) AS avg_position_month
    FROM march_facts
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
""").df()

monthly["position_tier"] = np.select(
    [monthly["avg_position_month"] <= 3, monthly["avg_position_month"] <= 10,
     monthly["avg_position_month"] <= 20, monthly["avg_position_month"] <= 50],
    ["top_3", "page_1", "striking", "page_3_5"], default="deep"
)

q = monthly[(monthly["impressions_month"] >= 100) & (monthly["avg_position_month"] >= 1)].copy()
q["ctr"] = q["clicks_month"] / q["impressions_month"]

content = con.sql(f"""
    SELECT content_hash_id, content_type, main_intent, is_published, is_deleted
    FROM read_parquet('{REL}/dim_content.parquet')
""").df()
q = q.merge(content, on="content_hash_id", how="inner")
q = q[(q["is_published"] == True) & (q["is_deleted"] == False)]

# stratum-size floor: fall back to tier-only average when the (tier, type) group is too thin to trust
MIN_STRATUM_N = 30
stratum_n = q.groupby(["position_tier", "content_type"])["ctr"].transform("count")
stratum_mean = q.groupby(["position_tier", "content_type"])["ctr"].transform("mean")
tier_mean = q.groupby("position_tier")["ctr"].transform("mean")
q["expected_ctr"] = np.where(stratum_n >= MIN_STRATUM_N, stratum_mean, tier_mean)
q["ctr_gap"] = q["expected_ctr"] - q["ctr"]

flagged = q[(q["position_tier"] != "deep") & (q["ctr_gap"] > 0)].copy()
flagged["score"] = flagged["ctr_gap"] * flagged["impressions_month"]
flagged["reason_code"] = "ctr_below_expected"
flagged["action"] = "review_title_meta"

queue = flagged.sort_values("score", ascending=False).reset_index(drop=True)
os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Pages considered: {len(q)}")
print(f"Flagged into the ranked queue: {len(queue)} ({len(queue)/len(q)*100:.1f}%)")
queue.head(10)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pages considered: 100702
Flagged into the ranked queue: 64274 (63.8%)


,client_hash_id,content_hash_id,impressions_month,clicks_month,avg_position_month,position_tier,ctr,content_type,main_intent,is_published,is_deleted,expected_ctr,ctr_gap,score,reason_code,action
0,client_73cda7b4e4f265ea,content_8e1334d6356668e3,134984.0,1.0,2.693038,top_3,0.000007,keyword article,commercial,True,False,0.003543,0.003535,477.212933,ctr_below_expected,review_title_meta
1,client_e547b89c05043229,content_8d7d99f109e19aa2,203497.0,289.0,2.468557,top_3,0.001420,keyword article,informational,True,False,0.003543,0.002123,431.936535,ctr_below_expected,review_title_meta
2,client_62f4a7e64f5e0096,content_34a70fea29d15f24,143019.0,43.0,3.166132,page_1,0.000301,keyword article,informational,True,False,0.003169,0.002868,410.169513,ctr_below_expected,review_title_meta
3,client_62f4a7e64f5e0096,content_7c6373141eae744a,132593.0,83.0,5.948459,page_1,0.000626,keyword article,commercial,True,False,0.003169,0.002543,337.133725,ctr_below_expected,review_title_meta
4,client_65de48885f4ef01b,content_62673eea26c31c17,57720.0,43.0,6.019317,page_1,0.000745,feedly article,None,True,False,0.006563,0.005818,335.836188,ctr_below_expected,review_title_meta
5,client_62f4a7e64f5e0096,content_f6116743b00afc2d,107584.0,15.0,9.735658,page_1,0.000139,keyword article,commercial,True,False,0.003169,0.003029,325.890294,ctr_below_expected,review_title_meta
6,client_62f4a7e64f5e0096,content_acbcc847f8996314,170808.0,262.0,3.396293,page_1,0.001534,keyword article,transactional,True,False,0.003169,0.001635,279.221644,ctr_below_expected,review_title_meta
7,client_9958f0a7ae1df715,content_cd3d932d4e1c8db0,89332.0,4.0,7.831807,page_1,0.000045,keyword article,informational,True,False,0.003169,0.003124,279.057069,ctr_below_expected,review_title_meta
8,client_a80fca3f171ed1de,content_046fc480045b88f5,83788.0,6.0,7.208276,page_1,0.000072,keyword article,informational,True,False,0.003169,0.003097,259.490370,ctr_below_expected,review_title_meta
9,client_62f4a7e64f5e0096,content_b99ea6861864dea5,194337.0,361.0,4.551516,page_1,0.001858,keyword article,informational,True,False,0.003169,0.001311,254.775552,ctr_below_expected,review_title_meta


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

1. **content_8e1334d6356668e3** — review_title_meta. keyword article, top_3
   (pos 2.7), 134,984 impressions, only 1 click. What would make it wrong:
   a single click on this much volume could be a tracking artifact  lowest
   confidence in the top 10.
2. **content_8d7d99f109e19aa2** — review_title_meta. keyword article, top_3
   (pos 2.5), 203,497 impressions, 289 clicks  real volume on both sides.
3. **content_34a70fea29d15f24** — review_title_meta. keyword article, page_1
   (pos 3.2), 143,019 impressions, 43 clicks.
4. **content_7c6373141eae744a** — review_title_meta. keyword article, page_1
   (pos 5.9), 132,593 impressions, 83 clicks.
5. **content_62673eea26c31c17** — review_title_meta. **feedly article**,
   page_1 (pos 6.0), 57,720 impressions, 43 clicks. Only non-keyword-article
   pick in the top 10 — its `main_intent` is missing (null), worth a manual
   look at the page before trusting the flag fully.
6. **content_f6116743b00afc2d** — review_title_meta. keyword article, page_1
   (pos 9.7), 107,584 impressions, only 15 clicks. What would make it wrong:
   thin click count, noisy CTR estimate.
7. **content_acbcc847f8996314** — review_title_meta. keyword article, page_1
   (pos 3.4), 170,808 impressions, 262 clicks  solid volume both sides.
8. **content_cd3d932d4e1c8db0** — review_title_meta. keyword article, page_1
   (pos 7.8), 89,332 impressions, only 4 clicks. What would make it wrong:
   same thin-click risk as #1 and #6.
9. **content_046fc480045b88f5** — review_title_meta. keyword article, page_1
   (pos 7.2), 83,788 impressions, 6 clicks. Same thin-click caveat.
10. **content_b99ea6861864dea5** — review_title_meta. keyword article,
    page_1 (pos 4.6), 194,337 impressions, 361 clicks  highest click count
    in the top 10, most trustworthy pick here.

**Pattern across the list:** one client (`client_62f4a7e64f5e0096`) accounts
for 5 of the 10 rows (#3, #4, #6, #7, #10). The rule has no per-client cap,
so a client with several large low-CTR pages can dominate. What would make
the whole queue wrong: if this reflects one client's unusual site structure
rather than 55+ clients' worth of independently discovered opportunities.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top10 = queue.head(10)[["content_hash_id", "content_type", "position_tier",
                         "avg_position_month", "impressions_month", "clicks_month",
                         "ctr", "expected_ctr", "score", "reason_code", "action"]]
top10["low_confidence"] = top10["clicks_month"] < 20
top10

,content_hash_id,content_type,position_tier,avg_position_month,impressions_month,clicks_month,ctr,expected_ctr,score,reason_code,action,low_confidence
0,content_8e1334d6356668e3,keyword article,top_3,2.693038,134984.0,1.0,0.000007,0.003543,477.212933,ctr_below_expected,review_title_meta,True
1,content_8d7d99f109e19aa2,keyword article,top_3,2.468557,203497.0,289.0,0.001420,0.003543,431.936535,ctr_below_expected,review_title_meta,False
2,content_34a70fea29d15f24,keyword article,page_1,3.166132,143019.0,43.0,0.000301,0.003169,410.169513,ctr_below_expected,review_title_meta,False
3,content_7c6373141eae744a,keyword article,page_1,5.948459,132593.0,83.0,0.000626,0.003169,337.133725,ctr_below_expected,review_title_meta,False
4,content_62673eea26c31c17,feedly article,page_1,6.019317,57720.0,43.0,0.000745,0.006563,335.836188,ctr_below_expected,review_title_meta,False
5,content_f6116743b00afc2d,keyword article,page_1,9.735658,107584.0,15.0,0.000139,0.003169,325.890294,ctr_below_expected,review_title_meta,True
6,content_acbcc847f8996314,keyword article,page_1,3.396293,170808.0,262.0,0.001534,0.003169,279.221644,ctr_below_expected,review_title_meta,False
7,content_cd3d932d4e1c8db0,keyword article,page_1,7.831807,89332.0,4.0,0.000045,0.003169,279.057069,ctr_below_expected,review_title_meta,True
8,content_046fc480045b88f5,keyword article,page_1,7.208276,83788.0,6.0,0.000072,0.003169,259.490370,ctr_below_expected,review_title_meta,True
9,content_b99ea6861864dea5,keyword article,page_1,4.551516,194337.0,361.0,0.001858,0.003169,254.775552,ctr_below_expected,review_title_meta,False


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks:** rows #1, #6, #8, #9 all have fewer than 20 clicks despite
six-figure impressions  noisy CTR estimates, confirmed below.

**Client concentration:** 5 of 10 top rows belong to one client, confirmed
below. **Small-stratum risk:** two (position_tier, content_type) groups
top_3×comparison and top_3×feedly — have only 9 pages each; their expected
CTR now falls back to the tier-only average instead of an unstable 9-row
mean. **Over-flagging:** 63.8% of considered pages score above zero  CTR is
right-skewed, so more than half of pages naturally sit below their stratum's
mean by construction, not because most of the panel is broken. This queue is
a ranking to work down from the top, not a "these are all genuinely bad" list.

**On precision@K:** not computable honestly yet  no independent future
outcome exists to check flags against. Checking against the rule's own
definition would be circular. Deferred to a later week once an outcome
label exists.

**Leakage check:** confirmed only `march_facts` (2026-03) and `dim_content`
were used — no forward window, no other month. No FlyRank product flags
(`needs_ctr_fix`, `is_quick_win`, `health_score`) exist in either table.

In [20]:
print("Client concentration in top 10:")
print(queue.head(10)["client_hash_id"].value_counts())

print("\nContent-type mix in top 10:")
print(queue.head(10)["content_type"].value_counts())

print("\nThin-click rows in top 10 (clicks_month < 20):")
print(queue.head(10)[queue.head(10)["clicks_month"] < 20][["content_hash_id", "impressions_month", "clicks_month"]])

print("\nSmall strata (n < 30) that received the tier-only fallback:")
print(q[q.groupby(["position_tier","content_type"])["ctr"].transform("count") < 30]
      .groupby(["position_tier","content_type"]).size())

print("\nOverall flag rate:", f"{len(queue)/len(q)*100:.1f}%")

Client concentration in top 10:
client_hash_id
client_62f4a7e64f5e0096    5
client_73cda7b4e4f265ea    1
client_e547b89c05043229    1
client_65de48885f4ef01b    1
client_9958f0a7ae1df715    1
client_a80fca3f171ed1de    1
Name: count, dtype: int64

Content-type mix in top 10:
content_type
keyword article    9
feedly article     1
Name: count, dtype: int64

Thin-click rows in top 10 (clicks_month < 20):
            content_hash_id  impressions_month  clicks_month
0  content_8e1334d6356668e3           134984.0           1.0
5  content_f6116743b00afc2d           107584.0          15.0
7  content_cd3d932d4e1c8db0            89332.0           4.0
8  content_046fc480045b88f5            83788.0           6.0

Small strata (n < 30) that received the tier-only fallback:
position_tier  content_type      
deep           comparison article    15
               feedly article        11
top_3          comparison article     9
               feedly article         9
dtype: int64

Overall flag rate: 63

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.